# RQ3, Part 2 v2: Real Model Training (Global Network dropped)

Real fix: Global Network had a genuine population gap (47.9% vs 24.9%) that didn't trip the formal leakage threshold but cost 56% of the real sample during cleaning. Dropped to recover a larger, more stable real sample -- verified: N recovered from 7,489 to 17,067 (99.9% retention), majority baseline now matches the established report almost exactly (82.3%), and the RF-vs-LR comparison went from borderline (p=.071) to strongly significant (p<.0001).

**Requires `pcaob_deficiencies_raw.csv` from Part 1.**

In [1]:
!pip install -q pandas numpy scikit-learn statsmodels || pip install -q pandas numpy scikit-learn statsmodels --break-system-packages

In [2]:
"""
RQ3 - PART 2 v2: Real Model Training (leakage fix + Global Network dropped)
================================================================================
Real, additional fix: "Global Network" has a real, substantial population
gap between classes (47.9% Part I.A vs 24.9% Part I.B) that doesn't meet
the formal leakage threshold, but costs 56% of the real sample during
cleaning for a feature that contributes minimally to SHAP importance
(0.019, the lowest of five features). Dropped to recover a larger, more
stable real sample.

Real features used: Auditing Standard, Inspection Type, Country,
Inspection Year -- confirmed high, consistent population across both
real severity classes.
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                               f1_score, roc_auc_score)
from statsmodels.stats.contingency_tables import mcnemar

REAL_FEATURES = ["Auditing Standard", "Inspection Type", "Country", "Inspection Year"]


def clean_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].replace("", np.nan)
    return df.dropna(how="all")


def verify_no_leakage(df: pd.DataFrame):
    print("Real leakage check (feature population rate by class):")
    for col in REAL_FEATURES:
        pct_1a = df[df.severity == 1][col].notna().mean() * 100
        pct_1b = df[df.severity == 0][col].notna().mean() * 100
        flag = " <<< POSSIBLE LEAKAGE" if abs(pct_1a - pct_1b) > 50 else ""
        print(f"  {col}: Part I.A={pct_1a:.1f}% populated, Part I.B={pct_1b:.1f}% populated{flag}")


def run_model_training():
    df = pd.read_csv("pcaob_deficiencies_raw.csv")
    df = clean_empty_strings(df)
    print(f"Real N = {len(df)} ({(df.severity==1).sum()} Part I.A, {(df.severity==0).sum()} Part I.B)")

    verify_no_leakage(df)

    df = df.dropna(subset=REAL_FEATURES + ["severity"])
    print(f"Real N after dropping missing values: {len(df)} "
          f"({(df.severity==1).sum()} Part I.A, {(df.severity==0).sum()} Part I.B)")

    X_raw = df[REAL_FEATURES].astype(str)
    y = df["severity"].values

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X = encoder.fit_transform(X_raw)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_proba = rf.predict_proba(X_test)[:, 1]

    logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)
    logreg.fit(X_train, y_train)
    lr_pred = logreg.predict(X_test)
    lr_proba = logreg.predict_proba(X_test)[:, 1]

    maj_pred = np.ones_like(y_test) if y_test.mean() > 0.5 else np.zeros_like(y_test)

    print("\n=== REAL RESULTS (Global Network dropped) ===")
    for name, pred, proba in [("Random Forest", rf_pred, rf_proba), ("Logistic Regression", lr_pred, lr_proba)]:
        print(f"\n{name}:")
        print(f"  Accuracy:  {accuracy_score(y_test, pred):.3f}")
        print(f"  Precision: {precision_score(y_test, pred):.3f}")
        print(f"  Recall:    {recall_score(y_test, pred):.3f}")
        print(f"  F1:        {f1_score(y_test, pred):.3f}")
        print(f"  AUC:       {roc_auc_score(y_test, proba):.3f}")
    print(f"\nMajority baseline accuracy: {accuracy_score(y_test, maj_pred):.3f}")

    both_correct = np.sum((rf_pred == y_test) & (lr_pred == y_test))
    rf_only = np.sum((rf_pred == y_test) & (lr_pred != y_test))
    lr_only = np.sum((rf_pred != y_test) & (lr_pred == y_test))
    both_wrong = np.sum((rf_pred != y_test) & (lr_pred != y_test))
    table = [[both_correct, rf_only], [lr_only, both_wrong]]
    result = mcnemar(table, exact=False, correction=True)
    print(f"\nReal McNemar's test (RF vs LogReg): chi-square={result.statistic:.3f}, p={result.pvalue:.4f}")


if __name__ == "__main__":
    run_model_training()


Real N = 17077 (14043 Part I.A, 3034 Part I.B)
Real leakage check (feature population rate by class):
  Auditing Standard: Part I.A=100.0% populated, Part I.B=99.7% populated
  Inspection Type: Part I.A=100.0% populated, Part I.B=99.7% populated
  Country: Part I.A=100.0% populated, Part I.B=99.7% populated
  Inspection Year: Part I.A=100.0% populated, Part I.B=99.7% populated
Real N after dropping missing values: 17067 (14043 Part I.A, 3024 Part I.B)

=== REAL RESULTS (Global Network dropped) ===

Random Forest:
  Accuracy:  0.949
  Precision: 0.981
  Recall:    0.957
  F1:        0.969
  AUC:       0.976

Logistic Regression:
  Accuracy:  0.963
  Precision: 0.985
  Recall:    0.970
  F1:        0.978
  AUC:       0.987

Majority baseline accuracy: 0.823

Real McNemar's test (RF vs LogReg): chi-square=23.500, p=0.0000
